In [4]:
import sys, os
import numpy as np
import pandas as pd
from scipy import stats

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tol_colors as tc

import scienceplots
plt.style.use(['science', 'no-latex'])
plt.rcParams['text.latex.preamble'] = r'\usepackage[cm]{sfmath}'
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'

current_dir = os.path.dirname(os.path.abspath('__file__'))

In [5]:
p = "no_preprompt"
model = "gemma-3-12b-it"
uncertainty_df = pd.read_csv(f"{current_dir}/data/{p}/{model}/uncertainty.csv")

In [6]:
factor = 10
potato_uncertainty = uncertainty_df[(uncertainty_df["dataset"]=="potato_final")]
overestimate_ids = potato_uncertainty[(potato_uncertainty["cs-hybrid"]>=factor*potato_uncertainty["oracle"])&(potato_uncertainty["n"]==10)]["id"]

print(f"Indices for which CS-Hybrid overestimates SE* by a factor of 10 or more: {overestimate_ids.values}")

Indices for which CS-Hybrid overestimates SE* by a factor of 10 or more: [  0   3  31  34  60  70  93 106 131]


In [7]:
num_sets_100_overestimate = []
num_sets_10_overestimate = []
se_100_overestimate = []
se_10_overestimate = []
for idx in overestimate_ids:
    num_sets_10 = potato_uncertainty[(potato_uncertainty["id"]==idx)&(potato_uncertainty["n"]==10)]["NumSets"].item()
    num_sets_100 = potato_uncertainty[(potato_uncertainty["id"]==idx)&(potato_uncertainty["n"]==100)]["NumSets"].item()
    
    num_sets_100_overestimate.append(num_sets_100)
    num_sets_10_overestimate.append(num_sets_10)

    se_10 = potato_uncertainty[(potato_uncertainty["id"]==idx)&(potato_uncertainty["n"]==10)]["se"].item()
    se_100 = potato_uncertainty[(potato_uncertainty["id"]==idx)&(potato_uncertainty["n"]==100)]["se"].item()
    
    se_100_overestimate.append(se_100)
    se_10_overestimate.append(se_10)

num_sets_100_overestimate = np.array(num_sets_100_overestimate)
num_sets_10_overestimate = np.array(num_sets_10_overestimate)
se_100_overestimate = np.array(se_100_overestimate)
se_10_overestimate = np.array(se_10_overestimate)

In [8]:
print("NumSets (n=100) for mega overestimates:", num_sets_100_overestimate)
print("NumSets (n=10) for mega overestimates:", num_sets_10_overestimate)

NumSets (n=100) for mega overestimates: [1 1 1 1 1 2 2 2 1]
NumSets (n=10) for mega overestimates: [1 1 1 1 1 2 2 2 1]


In the appendix, we consider dropping the worst offender. Let's check that dropping indices with SE^\*<0.005 does not inadvertently drop additional instances (it does not, since there is only one id resulting in 0<SE^\*<0.005):

In [9]:
potato_uncertainty[(potato_uncertainty["oracle"]<0.005)&(potato_uncertainty["NumSets"]>1)]["id"].unique()

array([93])

Now let's plot the modified figure

In [10]:
potato_duplicate_questions = [14, 83, 121]
models = [
    "gemma-2-9b-it",
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Phi-3.5-mini-instruct",
]

model_rename = {
    "gemma-2-9b-it": "Gemma-2-9B",
    "gemma-3-12b-it": "Gemma-3-12B",
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B",
    "Phi-3.5-mini-instruct": "Phi-3.5-3.8B",
}

entropy_methods = {
    "plugin": "$\\widehat{\\mathbb{H}}_{Plugin}$", # uses NumSets
    "cs": "$\\widehat{\\mathbb{H}}_{CS-GT}$", # uses GT
    "cs-hybrid": "$\\widehat{\\mathbb{H}}_{Hybrid}$", # uses H
}

datasets = {
    "hotpot_qa_final": "HotpotQA",
    "squad_v2_final": "SQuAD 2.0",
    "potato_final": "POTATO",
    "bioasq_final": "BioASQ",
}
pm_symbol = u"\u00B1"
num_samples_list = [5, 10, 25, 50, 75, 100]

In [ ]:
square = False

if square:
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 1])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
else:
    fig = plt.figure(figsize=(25, 5))
    gs = gridspec.GridSpec(1, 5, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[0, 3])
    ax5 = fig.add_subplot(gs[0, 4])
    
axes = [ax1, ax2, ax3, ax4, ax5]

colors = tc.get_colorset('muted')
dataset_colors = {
    "HotpotQA": colors[0],
    "SQuAD 2.0": colors[1],
    "POTATO": colors[3],
    "BioASQ": colors[4]
}

large_fontsize = 35
medium_fontsize = 30
small_fontsize = 25

for model_idx in range(len(models)):
    model = models[model_idx]
    ax = axes[model_idx]
    ax.axhline(
        y=1., 
        linestyle=':', 
        color='grey', 
        label='$\\langle\widehat{SE}/SE^*\\rangle=1$'
    )

    entropy_df = pd.read_csv(f"{current_dir}/data/{p}/{model}/uncertainty.csv")

    aggregate_means_hybrid = None
    aggregate_means_plugin = None
    aggregate_means_se = None

    for dataset in datasets:
        means_hybrid = []
        means_plugin = []
        means_se = []
        
        for n in num_samples_list:
            df_subset = entropy_df[(entropy_df["dataset"]==dataset)&(entropy_df["n"]==n)&(entropy_df["oracle"]>0.005)]
            ratios_hybrid = df_subset["cs-hybrid"]/df_subset["oracle"]
            ratios_plugin = df_subset["plugin"]/df_subset["oracle"]
            ratios_se = df_subset["se"]/df_subset["oracle"]

            means_hybrid.append(ratios_hybrid.mean())
            means_plugin.append(ratios_plugin.mean())
            means_se.append(ratios_se.mean())
        
        if aggregate_means_hybrid is None:
            aggregate_means_hybrid = np.array(means_hybrid)
        else:
            aggregate_means_hybrid += np.array(means_hybrid)

        if aggregate_means_plugin is None:
            aggregate_means_plugin = np.array(means_plugin)
        else:
            aggregate_means_plugin += np.array(means_plugin)

        if aggregate_means_se is None:
            aggregate_means_se = np.array(means_se)
        else:
            aggregate_means_se += np.array(means_se)

    aggregate_means_hybrid /= len(datasets)
    aggregate_means_plugin /= len(datasets)
    aggregate_means_se /= len(datasets)

    ax.plot(
        num_samples_list, aggregate_means_hybrid, 
        "o", markersize=12,
        label="$\\widehat{\\mathbb{H}}_{Hybrid}$ (Ours)",
        color=colors.indigo,
        linestyle=None,
        lw=4
    )

    ax.plot(
        num_samples_list, aggregate_means_plugin, 
        "x", markersize=12,
        label="$\\widehat{\\mathbb{H}}_{Plugin}$ (Canonical DSE)",
        color=colors.green,
        linestyle=":",
        lw=4
    )

    ax.plot(
        num_samples_list, aggregate_means_se, 
        "s", markersize=12,
        label="SE (White-Box)",
        color=colors.cyan,
        linestyle=":",
        lw=4
    )

    sns.despine(top=True, right=True, left=False, bottom=False, ax=ax)
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.tick_params(axis="y", which="both", right=False)
    ax.tick_params(axis="x", which="both", top=False) 

    ax.set_title(model_rename[model], fontsize=large_fontsize)
    ax.set_xlabel("$n$", fontsize=large_fontsize)
    if model_idx == 0:
        ax.set_ylabel("$\\langle\widehat{SE}/SE^*\\rangle$", fontsize=large_fontsize)
    ax.set_ylim(0.3, 1.2)
    ax.set_xscale("log")
    ax.set_xticks([5, 10, 25, 50, 100])
    ax.set_xticklabels([5, 10, 25, 50, 100])
    ax.set_yticks([0.4, 0.6, 0.8, 1.0, 1.2])
    ax.tick_params(axis='both', which='both', labelsize=small_fontsize)

handles, labels = ax.get_legend_handles_labels()
handles = [handles[1], handles[2], handles[3], handles[0]]
labels = [labels[1], labels[2], labels[3], labels[0]]
# interleave for 2 column setup
if square:
    handles = [handles[i] for i in range(1, len(handles), 2)]+[handles[i] for i in range(0, len(handles), 2)]
    labels = [labels[i] for i in range(1, len(labels), 2)]+[labels[i] for i in range(0, len(labels), 2)]
fig.legend(
    handles, labels, fontsize=medium_fontsize, loc='lower center', 
        bbox_to_anchor=(0.5, -0.3), ncol=4
)

plt.tight_layout()
plt.savefig('figures/entropy_ratios_potato_examination_draft.pdf')